# Customer-cone comparison

## 1. Purpose

**This notebook is an extension beyond the brief.** The brief's required
baseline for "who is big" is in-degree (notebook 02); everything here is
additional evidence, not a requirement.

The *customer cone* of an AS is the set of ASes reachable from it by
following provider-to-customer links downwards (Luckie et al., *AS
Relationships, Customer Cones, and Validation*, IMC 2013, cited in the
README). CAIDA ships it in the `ppdc-ases` file of the same snapshot
(`data/raw/20260901.ppdc-ases.bz2`), so it is an independent expert
ranking of the same graph on the same day, computed by the measurement
community that produced the topology. It measures the transit hierarchy,
whereas the PageRank top under peering `both` is dominated by heavily
peered networks (notebook 02, section 11).

That gives this notebook its question: **which peering variant brings
PageRank closer to the cone?** We compare the cone ranking against
PageRank under peering `both`, PageRank under peering `none`, and
in-degree, and report the tau-b of each against the cone. No decision on
peering is taken here; the number is the deliverable.

The computation lives in `pagerank`; this notebook reads, runs, tabulates
and plots.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pagerank import (
    from_edge_list,
    pagerank_power,
    ranking,
    rank_positions,
    rank_positions_ties,
    top_k_overlap,
    kendall_tau,
)
from pagerank.caida import (
    customer_cone_sizes,
    load_build,
    read_as_rel,
    read_ppdc_ases,
)

REPO = Path.cwd().parent
RAW = REPO / 'data' / 'raw'
PROCESSED = REPO / 'data' / 'processed'
FIGURES = REPO / 'results' / 'figures'
TABLES = REPO / 'results' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SNAPSHOT = '20260901'
AS_REL = RAW / f'{SNAPSHOT}.as-rel.bz2'
PPDC = RAW / f'{SNAPSHOT}.ppdc-ases.bz2'
PRIMARY_NPZ = PROCESSED / f'{SNAPSHOT}_peering-both_vote-c2p_ixp-kept.npz'
NONE_NPZ = PROCESSED / f'{SNAPSHOT}_peering-none_vote-c2p_ixp-kept.npz'

ALPHA = 0.85
TOL = 1e-12
t_start = time.perf_counter()

## 2. Reading the cones
`read_ppdc_ases` parses the file into one cone size per AS, counting the
distinct ASNs on each line including the leading AS itself. The header
carries `# total size:` (the number of ASes the file claims to cover) and
an `# inferred clique:` line, which is **empty** on this snapshot, so the
cone file does not itself name the clique; we take the 18-AS clique from
the `as-rel` header instead (the same set notebook 02 uses).

We print the line count against `total_size`, the measured self-listing
share, and the top 10 by cone size with a clique flag. The line count
reconciles (it equals `total_size`, asserted), but a line count is not
consistency: by definition a customer cone contains every direct customer,
so `cone_size - 1 >= customers` must hold for every AS, with `customers`
counted from the relationship file. The check below finds that it does
**not** hold everywhere. The starkest case is **AS286**, a clique member
the file gives cone size 1 while the relationships record 805 direct
customers; it sits alone at cone size 1 near the top of the scatter in
section 6. We keep the shipped cone as the reference ranking (it is CAIDA's
own number and the point of this extension) but record, quantified, that it
carries inconsistencies.

In [ ]:
cones = read_ppdc_ases(PPDC)
# Clique from the as-rel header (the ppdc clique line is blank on this
# snapshot). Re-reading the relationship file also gives the per-ASN customer
# and peer counts the consistency check below and the tables later reuse.
snap = read_as_rel(AS_REL)
clique_asns = np.array(snap.clique, dtype=np.int64)
clique_set = set(int(a) for a in clique_asns)
rel = snap.relationships
c2p = rel[rel[:, 2] == -1]
p2p = rel[rel[:, 2] == 0]
# A provider (as1 of a -1 row) gains one customer per such row; a peer appears
# as either endpoint of a 0 row.
n_customers_by_asn = pd.Series(c2p[:, 0]).value_counts()
peer_endpoints = np.concatenate([p2p[:, 0], p2p[:, 1]])
n_peers_by_asn = pd.Series(peer_endpoints).value_counts()
n_cone_lines = int(cones.asns.shape[0])
print(f'cone lines              {n_cone_lines}')
print(f'header total size       {cones.total_size}')
print(f'self-listed share       {cones.self_listed_share:.3f}')
print(f'comment lines           {cones.n_comment_lines}')
print(f'clique ASes (from as-rel): {len(clique_asns)}')
assert n_cone_lines == cones.total_size, (n_cone_lines, cones.total_size)
# cone_size counts distinct ASNs including the AS itself, so an AS with no
# customers has size 1 regardless of whether the file repeats its own ASN.
print(
    'cone size = distinct ASNs on the line incl. the AS itself; '
    f'min={int(cones.cone_size.min())}, max={int(cones.cone_size.max())}'
)

In [ ]:
cone_rank_all = rank_positions(cones.cone_size.astype(np.float64))
order_cone = ranking(cones.cone_size.astype(np.float64))
top10_cone = pd.DataFrame([
    {
        'cone_rank': int(cone_rank_all[i]),
        'asn': int(cones.asns[i]),
        'cone_size': int(cones.cone_size[i]),
        'clique': int(cones.asns[i]) in clique_set,
    }
    for i in order_cone[:10]
])
print(f'clique members in the cone top 10: {int(top10_cone["clique"].sum())} of 10')
top10_cone

In [ ]:
# Consistency check: a cone contains the AS's direct customers, so
# cone_size - 1 >= customers must hold for every AS. Count the violations
# and show the worst, with a clique flag, over the shipped cone's ASNs.
cone_customers = np.array(
    [int(n_customers_by_asn.get(int(a), 0)) for a in cones.asns]
)
deficit = cone_customers - (cones.cone_size - 1)  # positive => violation
violates = deficit > 0
n_viol = int(violates.sum())
n_clique_viol = sum(
    1 for i in np.where(violates)[0] if int(cones.asns[i]) in clique_set
)
print(f'line count == total size: {n_cone_lines == cones.total_size} '
      f'(reconciles)')
print(f'ASes with cone_size - 1 < customers: {n_viol} of {n_cone_lines}')
print(f'  of which clique members: {n_clique_viol} of {len(clique_asns)}')

worst = np.argsort(-deficit)[:10]
worst_df = pd.DataFrame([
    {
        'asn': int(cones.asns[i]),
        'cone_size': int(cones.cone_size[i]),
        'customers': int(cone_customers[i]),
        'clique': int(cones.asns[i]) in clique_set,
    }
    for i in worst
])
print('ten worst by customers - (cone_size - 1):')
worst_df

## 3. Alignment with the primary build

The rankings compared below live on the primary build's node indexing
(peering `both`, vote `customer_to_provider`, IXPs kept; 80,510 nodes).
We load it and build a `cone` score vector on those same indices: each
node gets its cone size from the file, or **1** if the file has no line
for it (an AS with no customers has cone size 1, so a missing line is
treated the same way). We report how many ASNs are only in the cone file
and how many only in the relationship file.

In [ ]:
build = load_build(PRIMARY_NPZ)
asns = build.asns
graph = from_edge_list(build.edges, build.n)

# Map cone sizes onto the primary node indexing. searchsorted against the
# sorted cone ASNs; nodes with no cone line default to size 1.
cone_vec = np.ones(graph.n, dtype=np.float64)
pos = np.searchsorted(cones.asns, asns)
clipped = np.clip(pos, 0, cones.asns.shape[0] - 1)
present = cones.asns[clipped] == asns
cone_vec[present] = cones.cone_size[clipped[present]].astype(np.float64)
n_missing_cone = int((~present).sum())

# ASNs only in the cone file: those not in the primary ASN set.
cone_only = np.setdiff1d(cones.asns, asns, assume_unique=False)
rel_only = np.setdiff1d(asns, cones.asns, assume_unique=False)

print(f'primary nodes                 {graph.n}')
print(f'cone lines                    {n_cone_lines}')
print(f'ASNs only in the cone file    {cone_only.shape[0]}')
print(f'ASNs only in the as-rel file  {rel_only.shape[0]} (cone treated as size 1)')
print(f'primary nodes with no cone line: {n_missing_cone}')
assert cone_vec.shape[0] == graph.n

## 4. Rankings compared against the cone

The three measures to set against the cone: PageRank under peering `both`
(the primary, recomputed here in about two seconds), PageRank under
peering `none`, and in-degree from the primary graph. For each, two kinds
of agreement with the cone: top-k overlap at 10/50/100, and Kendall's
tau-b over the union of the top 1000 by cone and the top 1000 by that
measure (the tail of size-1 cones would otherwise drown an all-node tau).

Higher cone size, higher PageRank and higher in-degree all mean "bigger",
so the score vectors go into the helpers directly.

In [ ]:
# Primary PageRank.
result = pagerank_power(graph, alpha=ALPHA, tol=TOL)
pr = result.vector
assert result.converged and abs(pr.sum() - 1.0) < 1e-12

# Peering-none PageRank, on the same node indexing.
none_build = load_build(NONE_NPZ)
assert np.array_equal(none_build.asns, asns)
none_graph = from_edge_list(none_build.edges, none_build.n)
none_result = pagerank_power(none_graph, alpha=ALPHA, tol=TOL)
pr_none = none_result.vector
assert none_result.converged

in_deg = graph.in_degree.astype(np.float64)

cone_top1000 = ranking(cone_vec)[:1000]


def vs_cone(score, name):
    union = np.union1d(cone_top1000, ranking(score)[:1000])
    return {
        'measure': name,
        'overlap@10': top_k_overlap(cone_vec, score, 10),
        'overlap@50': top_k_overlap(cone_vec, score, 50),
        'overlap@100': top_k_overlap(cone_vec, score, 100),
        'tau_b_top1000_union': kendall_tau(cone_vec, score, subset=union),
    }


summary = pd.DataFrame([
    vs_cone(pr, 'pagerank_both'),
    vs_cone(pr_none, 'pagerank_none'),
    vs_cone(in_deg, 'in_degree'),
])
summary_path = TABLES / '05_cone_vs_rankings_summary.csv'
summary.to_csv(summary_path, index=False)
print('wrote', summary_path.relative_to(REPO))
print(f'peering-none iterations: {none_result.iterations}')
summary

## 4b. Cross-check: cones recomputed from the relationships
The shipped cone is the reference, but how far does it sit from the cone the
relationship file itself implies? `customer_cone_sizes` recomputes each cone
directly: the distinct ASes reachable from an AS by following
provider-to-customer links downwards, by breadth-first search. We run it over
the union of the top 1000 by shipped cone and the top 1000 by PageRank, and
compare with the shipped sizes: how many match exactly, the mean absolute
relative difference, and the five largest disagreements. This quantifies the
inconsistency section 2 flagged; the shipped cone stays the reference for
every comparison that follows.

In [ ]:
union_cc = np.union1d(ranking(cone_vec)[:1000], ranking(pr)[:1000])
union_asns = asns[union_cc]
own_cone = customer_cone_sizes(snap, [int(a) for a in union_asns])
shipped_cone = cone_vec[union_cc].astype(np.int64)
exact = int((own_cone == shipped_cone).sum())
mean_abs_rel = float(np.mean(np.abs(own_cone - shipped_cone)
                             / np.maximum(shipped_cone, 1)))
print(f'union of the two top 1000s:        {union_cc.size} ASes')
print(f'own cone == shipped cone exactly:  {exact}')
print(f'mean abs relative difference:      {mean_abs_rel:.2f}')
worst = np.argsort(-np.abs(own_cone - shipped_cone))[:5]
cc_disagree = pd.DataFrame({
    'asn': union_asns[worst],
    'own_cone': own_cone[worst],
    'shipped_cone': shipped_cone[worst],
    'clique': [int(a) in clique_set for a in union_asns[worst]],
})
cc_disagree.reset_index(drop=True)

The question from section 1 is answered by the `tau_b_top1000_union`
column. Comparing the two PageRank rows tells us whether dropping peering
moves PageRank towards the transit-hierarchy ordering the cone measures:
the variant with the larger tau-b against the cone is the one closer to
it. The in-degree row is the brief's baseline for reference. The reading
of what this implies for the peering decision stays with the author; the
notebook reports the numbers and stops.

## 5. Top 20 by cone size

The 20 largest cones, each with its cone rank, the PageRank rank it holds
under both peering variants, its in-degree rank, its customer and peer
counts from the raw snapshot, and a clique flag. This shows where the
networks the cone calls biggest sit in the other orderings.

In [ ]:
# Per-ASN customer and peer counts were computed in section 2
# (n_customers_by_asn, n_peers_by_asn) and are reused here.
pr_rank = rank_positions(pr)
none_pr_rank = rank_positions(pr_none)
in_rank = rank_positions(in_deg)
cone_rank = rank_positions(cone_vec)

# Node order by cone size, over the primary node indexing.
order_cone_nodes = ranking(cone_vec)


def cone_row(node):
    asn = int(asns[node])
    return {
        'cone_rank': int(cone_rank[node]),
        'asn': asn,
        'cone_size': int(cone_vec[node]),
        'pagerank_both_rank': int(pr_rank[node]),
        'pagerank_none_rank': int(none_pr_rank[node]),
        'in_degree_rank': int(in_rank[node]),
        'customers': int(n_customers_by_asn.get(asn, 0)),
        'peers': int(n_peers_by_asn.get(asn, 0)),
        'clique': asn in clique_set,
    }


top20_cone = pd.DataFrame([cone_row(int(n)) for n in order_cone_nodes[:20]])
top20_path = TABLES / '05_top20_customer_cone.csv'
top20_cone.to_csv(top20_path, index=False)
print('wrote', top20_path.relative_to(REPO))
print(f'clique members in the cone top 20: {int(top20_cone["clique"].sum())} of 20')
top20_cone

## 6. PageRank against cone size

A log-log scatter of primary PageRank against cone size for every node,
clique members in a distinct colour, the top 5 by cone size annotated
with their ASN. The spread at a fixed cone size is the disagreement the
next section inspects.

In [ ]:
clique_nodes = np.array(
    [int(np.searchsorted(asns, a)) for a in clique_asns], dtype=np.int64
)
assert np.array_equal(asns[clique_nodes], clique_asns)
clique_mask = np.zeros(graph.n, dtype=bool)
clique_mask[clique_nodes] = True

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(cone_vec[~clique_mask], pr[~clique_mask], s=4, alpha=0.2,
           label='AS')
ax.scatter(cone_vec[clique_mask], pr[clique_mask], s=28, color='crimson',
           label='clique')

for node in ranking(cone_vec)[:5]:
    ax.annotate(str(int(asns[node])), (cone_vec[node], pr[node]),
                fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('customer-cone size')
ax.set_ylabel('PageRank (peering both)')
ax.set_title('PageRank vs customer-cone size')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
scatter_path = FIGURES / '05_cone_vs_pagerank.png'
fig.savefig(scatter_path, dpi=150)
print('wrote', scatter_path.relative_to(REPO))
plt.show()

## 7. Where the cone and PageRank disagree
Before the disagreement table, the headline that the comparison turns on:
how many of the PageRank top 20 / 100 / 1000 have **cone size 1**, i.e. no
inferred customers at all. These are the heavily peered networks peering
`both` lifts to the top while the cone, counting only customers, leaves at
the bottom.

For the shift itself the cone rank must be **tie-aware**. Most ASes (about
84%) have cone size 1, so an index-tie-broken rank would scatter them across
tens of thousands of arbitrary positions and the "largest shifts" would be
size-1 ASes that merely happen to have high node indices. `rank_positions_ties`
gives every tied cone the same (minimum) rank, so a shift reflects a real
difference in standing. We then take the ten largest tie-aware shifts each
way: `pagerank_higher` (PageRank ranks the AS well above the cone) and
`cone_higher` (the reverse).

For the mechanism we reuse the three in-neighbour quantities from notebook 02,
copied in rather than imported:
- `nbr_pr_rel`: mean PageRank of the in-neighbours over the graph-wide mean.
- `nbr_outdeg_median`: median out-degree of the in-neighbours.
- `contrib_per_link`: mean of `pr[tail] / out_degree[tail]` over the
  in-neighbours, the per-link term of the PageRank update itself.
The cone size of each AS is shown alongside.

In [ ]:
# Headline: PageRank top 20/100/1000 ASes with cone size 1 (no customers).
pr_order = ranking(pr)
for k in (20, 100, 1000):
    n_size1 = int(sum(1 for node in pr_order[:k] if cone_vec[node] == 1))
    print(f'PageRank top {k:>4}: {n_size1} with cone size 1 (no customers)')

# Tie-aware cone rank: all size-1 cones share one rank, so a shift is not a
# tie-breaking artefact. pr_rank stays the index-tie-broken rank from the
# ranking helper (PageRank scores are effectively all distinct).
cone_rank_ties = rank_positions_ties(cone_vec)

# In-neighbour summary, copied from notebook 02's section 6. transpose[head,
# tail] != 0 for an edge tail -> head, so row `node` of the stored transpose
# holds node's in-neighbours.
out_deg = graph.out_degree
pt_csr = graph.transpose.tocsr()
mean_pr = float(pr.mean())
def in_neighbour_stats(node):
    start, end = pt_csr.indptr[node], pt_csr.indptr[node + 1]
    tails = pt_csr.indices[start:end]
    if tails.size == 0:
        return np.nan, np.nan, np.nan
    nbr_pr_rel = float(pr[tails].mean() / mean_pr)
    nbr_outdeg_median = float(np.median(out_deg[tails]))
    contrib_per_link = float((pr[tails] / out_deg[tails]).mean())
    return nbr_pr_rel, nbr_outdeg_median, contrib_per_link

union_top = np.union1d(ranking(cone_vec)[:1000], ranking(pr)[:1000])
union_nodes = np.array(sorted(int(n) for n in union_top), dtype=np.int64)
# Signed shift using the tie-aware cone rank; negative when PageRank ranks the
# node better (smaller rank number) than the cone does.
signed_shift = pr_rank.astype(np.int64) - cone_rank_ties.astype(np.int64)
def rows_for_direction(candidate_nodes, direction, k=10):
    ordered = sorted(candidate_nodes,
                     key=lambda n: (-abs(int(signed_shift[n])), int(n)))
    out = []
    for node in ordered[:k]:
        nbr_pr_rel, nbr_outdeg_median, contrib_per_link = in_neighbour_stats(node)
        asn = int(asns[node])
        out.append({
            'asn': asn,
            'cone_rank': int(cone_rank_ties[node]),
            'cone_size': int(cone_vec[node]),
            'pagerank_rank': int(pr_rank[node]),
            'shift': abs(int(signed_shift[node])),
            'customers': int(n_customers_by_asn.get(asn, 0)),
            'peers': int(n_peers_by_asn.get(asn, 0)),
            'nbr_pr_rel': round(nbr_pr_rel, 1),
            'nbr_outdeg_median': round(nbr_outdeg_median, 1),
            'contrib_per_link': contrib_per_link,
            'clique': asn in clique_set,
            'direction': direction,
        })
    return out
promoted_nodes = [int(n) for n in union_nodes if signed_shift[n] < 0]
demoted_nodes = [int(n) for n in union_nodes if signed_shift[n] > 0]
promoted_df = pd.DataFrame(rows_for_direction(promoted_nodes, 'pagerank_higher'))
demoted_df = pd.DataFrame(rows_for_direction(demoted_nodes, 'cone_higher'))
disagreements = pd.concat([promoted_df, demoted_df], ignore_index=True)
disagreements_path = TABLES / '05_cone_disagreements.csv'
disagreements.to_csv(disagreements_path, index=False)
print('wrote', disagreements_path.relative_to(REPO))
disagreements

The structural facts, read off the columns above:

- The ASes **PageRank ranks above the cone** (`pagerank_higher`) are
  heavily peered networks: large `peers`, modest `customers`. The cone
  counts only customers reached through provider-to-customer links, so
  peering buys nothing in the cone ordering, while under peering `both`
  it buys in-links and lifts the PageRank rank.
- The ASes **the cone ranks above PageRank** (`cone_higher`) are transit
  providers with large `customers` and comparatively few `peers`: big
  cones, but under peering `both` their PageRank is diluted among the
  peering-driven networks above them.
- The `contrib_per_link` and `nbr_outdeg_median` columns show the same
  vote-dilution mechanism notebook 02 found against in-degree: the term
  `pr[tail] / out_degree[tail]` of the update, not a difference in how
  important the pointers are. The interpretation of which real networks
  these are stays with the author; the columns are the fact.

## 8. What this notebook established
- The cone file's line count reconciles with the header `# total size:`
  (asserted in section 2), but the file is **not fully consistent** with the
  relationships: 19 ASes have fewer cone members than direct customers, which
  a cone cannot. The worst is **AS286**, a clique member with cone size 1 in
  the file but 805 direct customers (PageRank rank 30). The shipped cone is
  kept as the reference anyway (section 4b quantifies how far it sits from the
  cone the relationships imply); this is a data-quality finding for the
  write-up, not a reason to change the reference.
- The cone is defined over the distinct ASNs on each line including the AS
  itself, so an AS with no customers has cone size 1; ASes present in the
  as-rel file but absent from the cone file are treated as size 1.
- A large share of the top PageRank ASes have **no customers at all** (cone
  size 1): 1 of the top 20, 5 of the top 100, 97 of the top 1000. Under
  peering `both` these are heavily peered networks the cone does not count,
  which is the mechanism behind the low agreement between PageRank `both` and
  the cone.
- The number that matters for the peering decision is the
  `tau_b_top1000_union` of each PageRank variant against the cone (section 4,
  `05_cone_vs_rankings_summary.csv`): peering `none` orders the networks far
  more like the transit hierarchy the cone measures than peering `both` does.
- Where the cone and primary PageRank disagree (with tie-aware cone ranks, so
  the table is not a size-1 artefact), the split is structural: PageRank lifts
  heavily peered networks the cone ignores, the cone lifts transit providers
  PageRank dilutes, by the same `pr[tail] / out_degree[tail]` mechanism as the
  in-degree comparison.
- This is an extension; no peering decision is taken here. The numbers are
  evidence for the author's modelling write-up.

In [ ]:
print(f'notebook wall time: {time.perf_counter() - t_start:.1f} s')